# Experiment 2: MLE vs Laplace Smoothing

This experiment investigates the zero-probability problem in
n-gram language models.

We compare:
- Maximum Likelihood Estimation (MLE)
- Laplace smoothing

The experiment focuses on unseen n-grams and their effect on
sentence probability and perplexity.

In [1]:
# PHASE 1: SETUP
import gzip
import json
import math

from ngram_lm import NGramLanguageModel

In [2]:
# PHASE 2: LOAD CORPUS

CORPUS_PATH = "C:/Users/ASUS/Downloads/c4-train.00000-of-01024-30K.json.gz"

documents = []

with gzip.open(CORPUS_PATH, "rt", encoding="utf-8") as file:
    for line in file:
        documents.append(json.loads(line))

print("Number of documents:", len(documents))

Number of documents: 30000


In [3]:
# Tokenize the corpus using the same procedure as Lab 1

def tokenize(text):
    text = text.lower()

    tokens = []
    current = []

    for ch in text:
        if ch.isalpha():
            current.append(ch)
        else:
            if current:
                tokens.append("".join(current))
                current = []

    if current:
        tokens.append("".join(current))

    return tokens


tokenized_documents = []

for document in documents:
    tokens = tokenize(document["text"])
    tokenized_documents.append(tokens)

print("Number of documents:", len(tokenized_documents))
print("Number of tokens:", sum(len(tokens) for tokens in tokenized_documents))

Number of documents: 30000
Number of tokens: 10876052


In [5]:
# PHASE 3: TRAIN MLE MODELS

unigram_model = NGramLanguageModel(1)
bigram_model = NGramLanguageModel(2)
trigram_model = NGramLanguageModel(3)

unigram_model.fit(tokenized_documents)
bigram_model.fit(tokenized_documents)
trigram_model.fit(tokenized_documents)

print("Unigram model trained.")
print("Bigram model trained.")
print("Trigram model trained.")

Unigram model trained.
Bigram model trained.
Trigram model trained.


In [6]:
# PHASE 4: TEST ZERO-PROBABILITY

observed_bigram = ["the", "cat"]

unseen_bigram = ["cat", "the"]

observed_probability = bigram_model.probability(
    ["the"],
    "cat"
)

unseen_probability = bigram_model.probability(
    ["cat"],
    "the"
)

print("Observed bigram:", observed_bigram)
print("MLE probability:", observed_probability)

print()

print("Unseen bigram:", unseen_bigram)
print("MLE probability:", unseen_probability)

Observed bigram: ['the', 'cat']
MLE probability: 0.00016119004306076865

Unseen bigram: ['cat', 'the']
MLE probability: 0.008645533141210375


In [7]:
# PHASE 4: FIND AN UNSEEN BIGRAM

candidate_words = [
    "the",
    "cat",
    "dog",
    "fish",
    "computer",
    "language",
    "student",
    "model",
    "university",
    "science"
]

unseen_context = None
unseen_word = None

for word1 in candidate_words:
    for word2 in candidate_words:
        if (word1, word2) not in bigram_model.bigram_counts:
            unseen_context = [word1]
            unseen_word = word2
            break

    if unseen_context is not None:
        break

print("Unseen context:", unseen_context)
print("Unseen word:", unseen_word)

print(
    "MLE probability:",
    bigram_model.probability(
        unseen_context,
        unseen_word
    )
)

Unseen context: ['cat']
Unseen word: dog
MLE probability: 0.0


In [8]:
# PHASE 5: APPLY LAPLACE SMOOTHING

laplace_probability = bigram_model.probability(
    unseen_context,
    unseen_word,
    smoothing=True
)

print("Unseen context:", unseen_context)
print("Unseen word:", unseen_word)
print("MLE probability:", 0.0)
print("Laplace probability:", laplace_probability)

Unseen context: ['cat']
Unseen word: dog
MLE probability: 0.0
Laplace probability: 5.621799088144188e-06


In [9]:
# PHASE 6: COMPARE MLE VS LAPLACE

test_sentence = [
    unseen_context[0],
    unseen_word
]

print("Test sentence:", test_sentence)

print()
print("Sentence probability:")

print(
    "MLE:",
    bigram_model.sentence_probability(test_sentence)
)

print(
    "Laplace:",
    bigram_model.sentence_probability(
        test_sentence,
        smoothing=True
    )
)

print()
print("Log probability:")

print(
    "MLE:",
    bigram_model.sentence_log_probability(test_sentence)
)

print(
    "Laplace:",
    bigram_model.sentence_log_probability(
        test_sentence,
        smoothing=True
    )
)

print()
print("Perplexity:")

print(
    "MLE:",
    bigram_model.perplexity(test_sentence)
)

print(
    "Laplace:",
    bigram_model.perplexity(
        test_sentence,
        smoothing=True
    )
)

Test sentence: ['cat', 'dog']

Sentence probability:
MLE: 0.0
Laplace: 3.5348471821062107e-10

Log probability:
MLE: -inf
Laplace: -21.763180861482173

Perplexity:
MLE: inf
Laplace: 53188.124713479905


In [10]:
# PHASE 7: RESULTS

mle_sentence_probability = bigram_model.sentence_probability(
    test_sentence
)

laplace_sentence_probability = bigram_model.sentence_probability(
    test_sentence,
    smoothing=True
)

mle_log_probability = bigram_model.sentence_log_probability(
    test_sentence
)

laplace_log_probability = bigram_model.sentence_log_probability(
    test_sentence,
    smoothing=True
)

mle_perplexity = bigram_model.perplexity(
    test_sentence
)

laplace_perplexity = bigram_model.perplexity(
    test_sentence,
    smoothing=True
)

print("=" * 60)
print("EXPERIMENT 2 RESULTS")
print("=" * 60)

print("Test sentence:", test_sentence)

print()
print("Sentence probability")
print("MLE:", mle_sentence_probability)
print("Laplace:", laplace_sentence_probability)

print()
print("Log probability")
print("MLE:", mle_log_probability)
print("Laplace:", laplace_log_probability)

print()
print("Perplexity")
print("MLE:", mle_perplexity)
print("Laplace:", laplace_perplexity)

EXPERIMENT 2 RESULTS
Test sentence: ['cat', 'dog']

Sentence probability
MLE: 0.0
Laplace: 3.5348471821062107e-10

Log probability
MLE: -inf
Laplace: -21.763180861482173

Perplexity
MLE: inf
Laplace: 53188.124713479905
